# Point Cloud Comparison
Load multiple `.ply` or `.npy` point clouds, downsample them for visualization, and compute pairwise Chamfer Distance on the original point clouds.

In [1]:
# Parameters
POINTCLOUD_FILES = [
    # Add paths to your .ply or .npy files here, e.g.:
    # "/home/ghr/fs/Junyi/data/proj/baseline_data/reconstructions_data/active-gs_procthor__ProcTHOR-Test-21_run_000/views_0004/gt_points_k.ply",
    # "/home/ghr/fs/Junyi/data/proj/baseline_data/reconstructions_data/active-gs_procthor__ProcTHOR-Test-21_run_000/views_0004/pred_points_aligned.ply",
    "/home/ghr/fs/Junyi/proj/baselines/reconstructions/random_replicacad__apt_1_run_000/views_0014/pred_points_aligned.ply",
    "/home/ghr/fs/Junyi/proj/baselines/reconstructions/random_replicacad__apt_1_run_000/views_0014/pred_points_aligned_cut3r.ply",
]

# Downsample: keep every N-th point for visualization (1 = no downsampling)
DOWNSAMPLE = 100

# Marker size in the 3D scatter plot
MARKER_SIZE = 2

# Compute pairwise Chamfer Distance for all loaded point clouds
COMPUTE_CD = True

# Maximum number of points used per cloud for CD (<= 0 means use all points)
CD_MAX_POINTS = 20000

# Seed for CD subsampling to make results reproducible
CD_RANDOM_SEED = 0

# If an input file has no colors, fall back to these distinct colors (hex)
FALLBACK_COLORS = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728",
    "#9467bd", "#8c564b", "#e377c2", "#7f7f7f",
]

In [2]:
import numpy as np
import plotly.graph_objects as go
import torch
import trimesh
from itertools import combinations
from pathlib import Path

In [3]:
def _normalize_colors(colors: np.ndarray | None) -> np.ndarray | None:
    if colors is None:
        return None
    colors = np.asarray(colors)
    if colors.ndim != 2 or colors.shape[1] < 3:
        return None
    colors = colors[:, :3]
    if np.issubdtype(colors.dtype, np.floating):
        finite = colors[np.isfinite(colors)]
        if finite.size and finite.max() <= 1.0:
            colors = colors * 255.0
    return np.clip(colors, 0, 255).astype(np.uint8)


def load_ply(path: str) -> tuple[np.ndarray, np.ndarray | None]:
    """Load a PLY file and return (points Nx3, colors Nx3 uint8 or None)."""
    mesh = trimesh.load(path, process=False)
    if isinstance(mesh, trimesh.PointCloud):
        pts = np.asarray(mesh.vertices, dtype=np.float32)
        colors = _normalize_colors(mesh.colors if mesh.colors is not None and len(mesh.colors) else None)
    elif isinstance(mesh, trimesh.Trimesh):
        pts = np.asarray(mesh.vertices, dtype=np.float32)
        if mesh.visual is not None and hasattr(mesh.visual, "vertex_colors") and mesh.visual.vertex_colors is not None:
            colors = _normalize_colors(mesh.visual.vertex_colors)
        else:
            colors = None
    else:
        raise ValueError(f"Unsupported geometry type: {type(mesh)} in {path}")
    return pts, colors


def load_npy(path: str) -> tuple[np.ndarray, np.ndarray | None]:
    """Load a NPY point cloud and return (points Nx3, colors Nx3 uint8 or None)."""
    arr = np.load(path, allow_pickle=True)

    if isinstance(arr, np.ndarray) and arr.dtype.names is not None:
        names = set(arr.dtype.names)
        if not {"x", "y", "z"}.issubset(names):
            raise ValueError(f"Structured NPY must contain x/y/z fields: {path}")
        pts = np.column_stack([arr["x"], arr["y"], arr["z"]]).astype(np.float32)
        if {"r", "g", "b"}.issubset(names):
            colors = _normalize_colors(np.column_stack([arr["r"], arr["g"], arr["b"]]))
        else:
            colors = None
        return pts, colors

    arr = np.asarray(arr)
    if arr.ndim != 2 or arr.shape[1] < 3:
        raise ValueError(
            f"NPY point cloud must be a 2D array with shape (N, 3+) or structured x/y/z fields: {path}, got {arr.shape}"
        )

    pts = np.asarray(arr[:, :3], dtype=np.float32)
    colors = _normalize_colors(arr[:, 3:6]) if arr.shape[1] >= 6 else None
    return pts, colors


def load_pointcloud(path: str) -> tuple[np.ndarray, np.ndarray | None]:
    suffix = Path(path).suffix.lower()
    if suffix == ".ply":
        return load_ply(path)
    if suffix == ".npy":
        return load_npy(path)
    raise ValueError(f"Unsupported point cloud format: {path}")


def _sample_points(points: np.ndarray, max_points: int, rng: np.random.Generator | None = None) -> np.ndarray:
    pts = np.asarray(points, dtype=np.float32).reshape(-1, 3)
    if max_points <= 0 or pts.shape[0] <= max_points:
        return pts
    if rng is None:
        rng = np.random.default_rng()
    idx = rng.choice(pts.shape[0], size=int(max_points), replace=False)
    return pts[idx]


def chamfer_distance(
    a: np.ndarray,
    b: np.ndarray,
    *,
    max_points: int = 20000,
    seed: int | None = 0,
    chunk_size: int = 2048,
) -> float | None:
    rng = np.random.default_rng(seed) if seed is not None else np.random.default_rng()
    a = _sample_points(a, max_points, rng)
    b = _sample_points(b, max_points, rng)
    if a.size == 0 or b.size == 0:
        return None

    a_t = torch.from_numpy(a)
    b_t = torch.from_numpy(b)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    a_t = a_t.to(device=device, dtype=torch.float32)
    b_t = b_t.to(device=device, dtype=torch.float32)

    mins_a = []
    for i in range(0, a_t.shape[0], chunk_size):
        dist = torch.cdist(a_t[i : i + chunk_size], b_t)
        mins_a.append(dist.min(dim=1).values)
    mins_a = torch.cat(mins_a, dim=0)

    mins_b = []
    for i in range(0, b_t.shape[0], chunk_size):
        dist = torch.cdist(b_t[i : i + chunk_size], a_t)
        mins_b.append(dist.min(dim=1).values)
    mins_b = torch.cat(mins_b, dim=0)

    return float(0.5 * (mins_a.mean() + mins_b.mean()).item())


def downsample(pts: np.ndarray, colors, stride: int):
    """Uniform stride downsampling for visualization only."""
    idx = np.arange(0, len(pts), stride)
    return pts[idx], (colors[idx] if colors is not None else None)


def make_trace(pts: np.ndarray, colors, name: str, fallback_color: str) -> go.Scatter3d:
    """Build a Scatter3d trace for one point cloud."""
    if colors is not None:
        color_arg = [f"rgb({r},{g},{b})" for r, g, b in colors]
    else:
        color_arg = fallback_color

    return go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode="markers",
        marker=dict(size=MARKER_SIZE, color=color_arg),
        name=name,
        hovertemplate="X: %{x:.3f}<br>Y: %{y:.3f}<br>Z: %{z:.3f}<extra>" + name + "</extra>",
    )

In [4]:
assert POINTCLOUD_FILES, "Please fill in POINTCLOUD_FILES in the Parameters cell above."

clouds = []
traces = []

for i, path in enumerate(POINTCLOUD_FILES):
    pts, colors = load_pointcloud(path)
    pts_ds, colors_ds = downsample(pts, colors, DOWNSAMPLE)
    name = Path(path).stem
    fallback = FALLBACK_COLORS[i % len(FALLBACK_COLORS)]

    clouds.append({
        "path": path,
        "name": name,
        "points": pts,
        "colors": colors,
    })
    traces.append(make_trace(pts_ds, colors_ds, name, fallback))
    print(
        f"[{i}] {name:40s}  total={len(pts):>8,}  shown={len(pts_ds):>7,}  colors={'yes' if colors is not None else 'no (fallback)'}"
    )

fig = go.Figure(data=traces)
fig.update_layout(
    title=f"Point Cloud Comparison  (downsample={DOWNSAMPLE}x, {len(POINTCLOUD_FILES)} clouds)",
    scene=dict(
        xaxis_title="X", yaxis_title="Y", zaxis_title="Z",
        aspectmode="data",
    ),
    legend=dict(itemsizing="constant"),
    width=1000, height=700,
)
fig.show()

if COMPUTE_CD:
    if len(clouds) < 2:
        print("\n[CD] Need at least two point clouds to compute Chamfer Distance.")
    else:
        print(
            f"\n[CD] Pairwise Chamfer Distance using original point clouds, max_points={CD_MAX_POINTS}, seed={CD_RANDOM_SEED}, device={'cuda' if torch.cuda.is_available() else 'cpu'}"
        )
        cd_matrix = np.full((len(clouds), len(clouds)), np.nan, dtype=np.float64)
        np.fill_diagonal(cd_matrix, 0.0)
        for i, j in combinations(range(len(clouds)), 2):
            cd = chamfer_distance(
                clouds[i]["points"],
                clouds[j]["points"],
                max_points=CD_MAX_POINTS,
                seed=CD_RANDOM_SEED,
            )
            cd_matrix[i, j] = cd
            cd_matrix[j, i] = cd
            print(f"[CD] {i}:{clouds[i]['name']}  <->  {j}:{clouds[j]['name']}  =  {cd:.6f}" if cd is not None else f"[CD] {i}:{clouds[i]['name']}  <->  {j}:{clouds[j]['name']}  =  None")

        print("\n[CD] Matrix")
        header = " " * 18 + " ".join(f"{i:^12d}" for i in range(len(clouds)))
        print(header)
        for i, cloud in enumerate(clouds):
            row = " ".join(f"{value:12.6f}" for value in cd_matrix[i])
            print(f"{i}:{cloud['name'][:14]:<16} {row}")

[0] pred_points_aligned                       total=3,756,536  shown= 37,566  colors=yes
[1] pred_points_aligned_cut3r                 total=2,752,512  shown= 27,526  colors=yes



[CD] Pairwise Chamfer Distance using original point clouds, max_points=20000, seed=0, device=cuda
[CD] 0:pred_points_aligned  <->  1:pred_points_aligned_cut3r  =  0.278471

[CD] Matrix
                       0            1      
0:pred_points_al       0.000000     0.278471
1:pred_points_al       0.278471     0.000000
